# New York City Taxi Trips
This notebook is working on a simple Extract, Transform, Load (ETL) pipeline design for public data of NYC taxi trips. The data was downloaded in the form of parquet files and then transformed by utilizing the library [PySpark](https://spark.apache.org/docs/latest/api/python/index.html). The transformed data is then ingested to [BigQuery](https://console.cloud.google.com/bigquery) as the data warehouse, where it is finally available to be used for analytical purposes such as analysis, reporting and dashboarding.

*Source*: [NYC Government TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

### 1. Import Necessary Libraries

In [7]:
from google.cloud import bigquery
import pandas as pd
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, to_timestamp, when, create_map, lit

### 2. Using the Library Pandas to do Exploratory Data Analysis (EDA)

In [17]:
df_pandas = pd.DataFrame()
files = os.listdir('/home/jasonzelin/data-analytics-portfolio/new_york_taxi_trips/data')
for file in files:
    if file.startswith('green_tripdata_'):
        df_loop = pd.read_parquet(f'/home/jasonzelin/data-analytics-portfolio/new_york_taxi_trips/data/{file}')
        df_pandas = pd.concat([df_pandas, df_loop])

df_pandas

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-03-01 00:07:34,2025-03-01 00:24:52,N,1.0,75,239,1.0,2.20,18.40,...,0.5,5.91,0.00,NaN,1.0,29.56,1.0,1.0,2.75,0.00
1,2,2025-03-01 00:01:24,2025-03-01 00:10:03,N,1.0,41,42,1.0,1.06,8.60,...,0.5,3.33,0.00,NaN,1.0,14.43,1.0,1.0,0.00,0.00
2,2,2025-03-01 00:45:03,2025-03-01 01:05:38,N,1.0,265,56,1.0,18.91,69.50,...,0.5,0.00,0.00,NaN,1.0,72.00,2.0,1.0,0.00,0.00
3,2,2025-03-01 00:10:10,2025-03-01 00:29:35,N,5.0,82,236,1.0,8.36,30.17,...,0.5,3.35,6.94,NaN,1.0,44.71,1.0,1.0,2.75,0.00
4,2,2025-03-01 00:16:14,2025-03-01 00:19:44,N,1.0,66,33,1.0,0.82,5.80,...,0.5,0.00,0.00,NaN,1.0,8.30,2.0,1.0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46616,2,2025-02-28 22:35:00,2025-02-28 23:10:00,None,NaN,181,161,NaN,8.23,44.39,...,0.5,7.41,0.00,NaN,1.0,56.80,NaN,NaN,NaN,0.75
46617,2,2025-02-28 23:40:00,2025-02-28 23:55:00,None,NaN,166,141,NaN,4.10,27.67,...,0.5,6.38,0.00,NaN,1.0,38.30,NaN,NaN,NaN,0.00
46618,2,2025-02-28 23:34:00,2025-02-28 23:48:00,None,NaN,41,48,NaN,4.09,27.71,...,0.5,0.00,0.00,NaN,1.0,32.71,NaN,NaN,NaN,0.75
46619,2,2025-02-28 23:52:00,2025-03-01 00:05:00,None,NaN,75,140,NaN,2.25,16.64,...,0.5,1.00,0.00,NaN,1.0,21.89,NaN,NaN,NaN,0.00


In [19]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146486 entries, 0 to 46620
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               146486 non-null  int32         
 1   lpep_pickup_datetime   146486 non-null  datetime64[us]
 2   lpep_dropoff_datetime  146486 non-null  datetime64[us]
 3   store_and_fwd_flag     138584 non-null  object        
 4   RatecodeID             138584 non-null  float64       
 5   PULocationID           146486 non-null  int32         
 6   DOLocationID           146486 non-null  int32         
 7   passenger_count        138584 non-null  float64       
 8   trip_distance          146486 non-null  float64       
 9   fare_amount            146486 non-null  float64       
 10  extra                  146486 non-null  float64       
 11  mta_tax                146486 non-null  float64       
 12  tip_amount             146486 non-null  float64   

### 3. Starting a Spark Session to Begin Transforming Data

In [24]:
# Start a Spark session
spark = SparkSession.builder \
    .appName("NYC Taxi ETL") \
    .getOrCreate()

# Limiting output log to just errors
spark.sparkContext.setLogLevel("ERROR")

In [25]:
df = spark.read.parquet('/home/jasonzelin/data-analytics-portfolio/new_york_taxi_trips/data/green_tripdata_*.parquet')
df.show(n=5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-03-01 00:07:34|  2025-03-01 00:24:52|                 N|         1|          75|         2

Converting timestamp columns and standardizing column nomenclature

In [46]:
df_clean = (
    df.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")  # Rename for clarity
      .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")  # Rename for clarity
      .withColumn("pickup_datetime", to_timestamp(col("pickup_datetime")))  # Convert to timestamp
      .withColumn("dropoff_datetime", to_timestamp(col("dropoff_datetime")))  # Convert to timestamp
      .withColumnRenamed("VendorID", "vendor_id")  # Rename for clarity
      .withColumnRenamed("RatecodeID", "rate_code_id")  # Rename for clarity
      .withColumnRenamed("PULocationID", "pickup_location_id")  # Rename for clarity
      .withColumnRenamed("DOLocationID", "dropoff_location_id")  # Rename for clarity
      .withColumnRenamed("payment_type", "payment_type_id")  # Rename for clarity
      .withColumnRenamed("trip_type", "trip_type_id")  # Rename for clarity
)
df_clean.show(n=5)

+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+---------------+------------+--------------------+------------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|rate_code_id|pickup_location_id|dropoff_location_id|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type_id|trip_type_id|congestion_surcharge|cbd_congestion_fee|
+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+---------------+------------+--------------------+------------------+
|        2|2025-03-01 00:07:34|2025-03-01 00:24:5

In [47]:
vendor_map_expr = create_map(
    [lit(1), lit("Creative Mobile Technologies, LLC"),
     lit(2), lit("Curb Mobility, LLC"),
     lit(6), lit("Myle Technologies Inc")
    ]
)

rate_code_map_expr = create_map(
    [lit(1), lit("Standard rate"),
     lit(2), lit("JFK"),
     lit(3), lit("Newark"),
     lit(4), lit("Nassau or Westchester"),
     lit(5), lit("Negotiated fare"),
     lit(6), lit("Group ride"),
     lit(99), lit("Unknown")
    ]
)

payment_type_map_expr = create_map(
    [lit(0), lit("flex_fare_trip"),
    lit(1), lit("credit_card"),
    lit(2), lit("cash"),
    lit(3), lit("no_charge"),
    lit(4), lit("dispute"),
    lit(5), lit("unknown"),
    lit(6), lit("voided_trip")
    ]
)

trip_type_map_expr = create_map(
    [lit(1), lit("street_hail"),
    lit(2), lit("dispatch")
    ]
)

location_mapping = spark.read.csv(
    '/home/jasonzelin/data-analytics-portfolio/new_york_taxi_trips/data/taxi_zone_lookup.csv',
    header=True,
    inferSchema=True
)


pickup_location_mapping = location_mapping.selectExpr(
    "LocationID as pickup_location_id",
    "concat(Zone, ', ', Borough) as pickup_location"
)

dropoff_location_mapping = location_mapping.selectExpr(
    "LocationID as dropoff_location_id",
    "concat(Zone, ', ', Borough) as dropoff_location"
)

df_clean = df_clean \
    .withColumn("vendor_name", vendor_map_expr.getItem(col("vendor_id"))) \
    .withColumn("rate_code", rate_code_map_expr.getItem(col("rate_code_id"))) \
    .withColumn("payment_type", payment_type_map_expr.getItem(col("payment_type_id"))) \
    .withColumn("trip_type", trip_type_map_expr.getItem(col("trip_type_id"))) \
    .join(pickup_location_mapping, on="pickup_location_id", how="left") \
    .join(dropoff_location_mapping, on="dropoff_location_id", how="left") \
    .drop("vendor_id") \
    .drop("rate_code_id") \
    .drop("payment_type_id") \
    .drop("trip_type_id") \
    .drop("dropoff_location_id") \
    .drop("pickup_location_id") \
    .drop("ehail_fee")  # Dropping column with too many nulls

print(df_clean.show(n=5))
print(df_clean.printSchema())

+-------------------+-------------------+------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+------------------+------------------+---------------+------------+-----------+--------------------+--------------------+
|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|cbd_congestion_fee|       vendor_name|      rate_code|payment_type|  trip_type|     pickup_location|    dropoff_location|
+-------------------+-------------------+------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+------------------+------------------+---------------+------------+-----------+--------------------+--------------------+
|2025-03-01 00:07:34|2025-03-01 00:24:52|     

In [58]:
# Exporting the cleaned data to a single parquet file
cleaned_file_path = '/home/jasonzelin/data-analytics-portfolio/new_york_taxi_trips/data/cleaned_nyc_taxi_data.parquet'
df_clean.coalesce(1).write.mode('overwrite').parquet(cleaned_file_path)

### 4. Ingesting the Transformed Data in Form of Parquet to the Data Warehouse (BigQuery Tables)

In [59]:
client = bigquery.Client.from_service_account_json("/home/jasonzelin/data-analytics-portfolio/jasonzelin-data-analytics-b2e7ad6f2da9.json")

In [63]:
table_id = 'jasonzelin-data-analytics.new_york_taxi_trips.green_taxi_trips'
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

for file in os.listdir(cleaned_file_path):
    if file.endswith('.parquet'):
        with open(f'{cleaned_file_path}/{file}', 'rb') as source_file:
            job = client.load_table_from_file(source_file, table_id, job_config=job_config)
        job.result()  # Waits for the job to complete.
        print(f'Loaded {file} into {table_id}.')

Loaded part-00000-1f48187c-2d60-41d8-b463-9dd8964944b1-c000.snappy.parquet into jasonzelin-data-analytics.new_york_taxi_trips.green_taxi_trips.


In [65]:
# Query the table to verify the data ingestion
client.query(f'select * from {table_id} order by pickup_datetime limit 5').to_dataframe()

/home/jasonzelin/data-analytics-portfolio/venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pickup_datetime,dropoff_datetime,store_and_fwd_flag,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,cbd_congestion_fee,vendor_name,rate_code,payment_type,trip_type,pickup_location,dropoff_location
0,2024-12-25 16:13:15+00:00,2024-12-25 16:13:17+00:00,N,3,0.00,35.0,0.0,0.0,7.20,0.00,1.0,43.20,0.00,0.0,"Curb Mobility, LLC",Negotiated fare,credit_card,dispatch,"Astoria, Queens","N/A, Unknown"
1,2024-12-28 18:23:41+00:00,2024-12-28 18:41:08+00:00,N,6,10.15,65.0,0.0,0.0,0.00,6.94,1.0,72.94,0.00,0.0,"Curb Mobility, LLC",Negotiated fare,cash,dispatch,"Flushing, Queens","East Harlem South, Manhattan"
2,2024-12-31 12:56:14+00:00,2024-12-31 13:03:33+00:00,N,1,1.00,8.6,1.0,0.5,2.22,0.00,1.0,13.32,0.00,0.0,"Curb Mobility, LLC",Standard rate,credit_card,street_hail,"Elmhurst, Queens","Jackson Heights, Queens"
3,2024-12-31 15:42:13+00:00,2024-12-31 15:42:31+00:00,N,1,0.06,23.0,1.0,0.0,0.00,0.00,1.0,25.00,0.00,0.0,"Curb Mobility, LLC",Newark,cash,street_hail,"East Harlem North, Manhattan","East Harlem North, Manhattan"
4,2024-12-31 16:01:11+00:00,2024-12-31 16:04:29+00:00,N,1,0.66,5.8,1.0,0.5,2.76,0.00,1.0,13.81,2.75,0.0,"Curb Mobility, LLC",Standard rate,credit_card,street_hail,"East Harlem South, Manhattan","Upper East Side North, Manhattan"
